# Student Management Portal
## Assignment 3 — End-to-End Portal Design

| Item | Detail |
|------|--------|
| **Portal Type** | Student Management Portal |
| **Language** | Python 3 (standard library only) |
| **Storage** | CSV for student records, JSON for user credentials |
| **Console file** | `portal.py` |
| **Web file** | `app.py` (Flask) |

---
### How to Run
- Run all cells top to bottom using **Kernel → Restart & Run All**.
- Demo cells at the bottom show each feature without needing `input()`.
- For the full interactive version: `python portal.py` in a terminal.
- For the web version: `python app.py` then open `http://localhost:5000`.

---
## Section 1: Imports and File Paths

We use only Python's built-in modules — no external libraries needed for the core logic.

- `json` — read and write the users file
- `csv` — read and write the students file
- `os` — check if folders exist and create them

In [1]:
import json
import csv
import os

# File paths
USERS_FILE   = "data/users.json"
RECORDS_FILE = "data/students.csv"
REPORT_FILE  = "reports/summary_report.txt"

# Column headers for the CSV
HEADERS = ["student_id", "name", "age", "course", "marks", "grade"]

print("Imports loaded successfully.")

Imports loaded successfully.


---
## Section 2: Helper Functions

Small utility functions used by the main features:

| Function | Purpose |
|----------|---------|
| `calculate_grade(marks)` | Converts a number to a letter grade |
| `load_data(filepath)` | Reads the CSV file into a list of dictionaries |
| `save_data(filepath, students)` | Writes the list back to the CSV file |
| `print_table(students)` | Displays records in a formatted table |

In [2]:
def calculate_grade(marks):
    """Return A, B, C, D or F based on marks out of 100."""
    if marks >= 80:
        return "A"
    elif marks >= 65:
        return "B"
    elif marks >= 50:
        return "C"
    elif marks >= 40:
        return "D"
    else:
        return "F"


def load_data(filepath):
    """Read student records from CSV. Returns a list of dicts."""
    students = []
    try:
        with open(filepath, "r") as file:
            reader = csv.DictReader(file)
            for row in reader:
                students.append(row)
    except FileNotFoundError:
        print(f"Note: '{filepath}' not found. Starting with empty data.")
        save_data(filepath, [])
    except PermissionError:
        print(f"Error: Cannot read '{filepath}'.")
    return students


def save_data(filepath, students):
    """Write the full students list to CSV."""
    try:
        folder = os.path.dirname(filepath)
        if folder and not os.path.exists(folder):
            os.makedirs(folder)
        with open(filepath, "w", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=HEADERS)
            writer.writeheader()
            for student in students:
                writer.writerow(student)
    except PermissionError:
        print(f"Error: Cannot write to '{filepath}'.")


def print_table(students):
    """Print student records as a formatted table."""
    print(f"  {'ID':<8} {'Name':<20} {'Age':<5} {'Course':<20} {'Marks':<7} Grade")
    print("  " + "-" * 68)
    for s in students:
        print(f"  {s['student_id']:<8} {s['name']:<20} {s['age']:<5} {s['course']:<20} {s['marks']:<7} {s['grade']}")


print("Helper functions defined.")

Helper functions defined.


---
## Section 3: FR-1 — User Login

- Reads credentials from `data/users.json`
- Loops up to 3 attempts using `while`
- Rejects blank input with a warning
- Catches `FileNotFoundError` and `json.JSONDecodeError`
- Returns `(username, role)` on success, `(None, None)` on failure

In [3]:
def user_login():
    """Check username and password against users.json. Max 3 attempts."""
    print("\n" + "=" * 40)
    print("        STUDENT PORTAL - LOGIN")
    print("=" * 40)

    try:
        with open(USERS_FILE, "r") as file:
            users = json.load(file)
    except FileNotFoundError:
        print("  Error: User file not found.")
        return None, None
    except json.JSONDecodeError:
        print("  Error: User file is corrupted.")
        return None, None

    attempts = 0
    max_attempts = 3

    while attempts < max_attempts:
        username = input("  Username: ").strip()
        password = input("  Password: ").strip()

        if not username or not password:
            print("  Username and password cannot be blank.\n")
            attempts += 1
            continue

        for user in users:
            if user["username"] == username and user["password"] == password:
                print(f"\n  Login successful. Welcome, {username}! Role: {user['role']}")
                return username, user["role"]

        attempts += 1
        left = max_attempts - attempts
        if left > 0:
            print(f"  Wrong credentials. {left} attempt(s) remaining.\n")
        else:
            print("  Too many failed attempts. Access denied.")

    return None, None


print("user_login() defined.")

user_login() defined.


---
## Section 4: FR-2 — Add Student Record

- Checks for duplicate IDs using a `for` loop
- Validates age with `try/except ValueError` — rejects non-numeric input
- Validates marks with `try/except ValueError`
- Calls `calculate_grade()` automatically
- Appends to list and saves to CSV

In [4]:
def add_record(students):
    """Collect and validate a new student, then save to file."""
    print("\n--- Add New Student ---")

    student_id = input("  Student ID: ").strip().upper()
    if not student_id:
        print("  Error: Student ID cannot be blank.")
        return students

    for s in students:
        if s["student_id"] == student_id:
            print(f"  Error: ID '{student_id}' already exists.")
            return students

    name = input("  Full Name: ").strip()
    if not name:
        print("  Error: Name cannot be blank.")
        return students

    try:
        age = int(input("  Age: ").strip())
        if age < 10 or age > 80:
            print("  Error: Age must be between 10 and 80.")
            return students
    except ValueError:
        print("  Error: Age must be a whole number.")
        return students

    course = input("  Course: ").strip()
    if not course:
        print("  Error: Course cannot be blank.")
        return students

    try:
        marks = float(input("  Marks (0-100): ").strip())
        if marks < 0 or marks > 100:
            print("  Error: Marks must be between 0 and 100.")
            return students
    except ValueError:
        print("  Error: Marks must be a number like 75 or 88.5.")
        return students

    grade = calculate_grade(marks)
    new_student = {
        "student_id": student_id,
        "name":       name,
        "age":        str(age),
        "course":     course,
        "marks":      str(marks),
        "grade":      grade
    }
    students.append(new_student)
    save_data(RECORDS_FILE, students)
    print(f"  Done! '{name}' added with grade {grade}.")
    return students


print("add_record() defined.")

add_record() defined.


---
## Section 5: FR-3 — Search Student

- Search by ID (exact match) or name (partial match)
- Uses a `for` loop to scan all records
- Case-insensitive name search using `.lower()`

In [5]:
def search_record(students):
    """Search by student ID (exact) or name (partial match)."""
    print("\n--- Search Student ---")
    print("  1. By Student ID")
    print("  2. By Name")
    choice = input("  Choose (1 or 2): ").strip()

    if choice == "1":
        key   = input("  Enter Student ID: ").strip().upper()
        field = "student_id"
    elif choice == "2":
        key   = input("  Enter name: ").strip().lower()
        field = "name"
    else:
        print("  Invalid choice.")
        return

    results = []
    for s in students:
        if field == "student_id" and s["student_id"] == key:
            results.append(s)
        elif field == "name" and key in s["name"].lower():
            results.append(s)

    if results:
        print(f"\n  Found {len(results)} record(s):\n")
        print_table(results)
    else:
        print("  No matching student found.")


print("search_record() defined.")

search_record() defined.


---
## Section 6: FR-4 — Generate Report

- Loops through all students to collect marks, grades, courses
- Calculates total, average, highest, lowest marks
- Counts students per grade and per course
- Saves report to `reports/summary_report.txt`
- Uses `finally` to always confirm completion

In [6]:
def generate_report(students):
    """Calculate summary stats and save report to file."""
    print("\n--- Summary Report ---")
    if not students:
        print("  No records to report on.")
        return

    total        = len(students)
    marks_list   = []
    grade_count  = {}
    course_count = {}

    for s in students:
        try:
            marks_list.append(float(s["marks"]))
        except ValueError:
            pass
        grade_count[s["grade"]]   = grade_count.get(s["grade"], 0) + 1
        course_count[s["course"]] = course_count.get(s["course"], 0) + 1

    avg = sum(marks_list) / len(marks_list) if marks_list else 0

    lines = [
        "=" * 45,
        "      STUDENT MANAGEMENT PORTAL REPORT",
        "=" * 45,
        f"  Total Students  : {total}",
        f"  Average Marks   : {avg:.2f}",
        f"  Highest Marks   : {max(marks_list):.2f}",
        f"  Lowest Marks    : {min(marks_list):.2f}",
        "",
        "  Grade Distribution:"
    ]
    for g in sorted(grade_count):
        lines.append(f"    Grade {g}: {grade_count[g]} student(s)")
    lines.append("")
    lines.append("  Students Per Course:")
    for c, count in course_count.items():
        lines.append(f"    {c}: {count} student(s)")
    lines.append("=" * 45)

    for line in lines:
        print(line)

    try:
        if not os.path.exists("reports"):
            os.makedirs("reports")
        with open(REPORT_FILE, "w") as f:
            for line in lines:
                f.write(line + "\n")
        print(f"\n  Report saved to '{REPORT_FILE}'.")
    except PermissionError:
        print("  Could not save report file.")
    finally:
        print("  Report generation complete.")


print("generate_report() defined.")

generate_report() defined.


---
## Section 7: Additional Functions — View, Update, Delete, Menu

These functions complete the CRUD workflow. They are defined here but, like the
interactive features above, they rely on `input()`, so the demo cells in
Section 8 exercise the same logic directly instead of calling them.

| Function | Purpose | Key detail |
|----------|---------|------------|
| `view_all_records(students)` | Prints every record in a formatted table | Shows a friendly message if the list is empty |
| `update_record(students)` | Finds a student by ID and edits their details | Pressing Enter keeps the current value; invalid numbers are ignored with a warning |
| `delete_record(students)` | Finds a student by ID and deletes them | Requires an explicit `yes` confirmation before removal |
| `display_menu()` | Prints the numbered menu (options 1–7) | Called by `main()` at the top of every loop iteration |

In [7]:
def view_all_records(students):
    """Print all student records in a table."""
    print("\n--- All Student Records ---")
    if not students:
        print("  No records found.")
        return
    print(f"  Total: {len(students)} student(s)\n")
    print_table(students)


def update_record(students):
    """Find a student by ID and update their details."""
    print("\n--- Update Student Record ---")
    student_id = input("  Student ID to update: ").strip().upper()
    found = None
    for s in students:
        if s["student_id"] == student_id:
            found = s
            break
    if not found:
        print(f"  Error: ID '{student_id}' not found.")
        return students
    print(f"  Updating: {found['name']} (press Enter to keep current value)")
    new_name = input(f"  Name [{found['name']}]: ").strip()
    if new_name:
        found["name"] = new_name
    new_age = input(f"  Age [{found['age']}]: ").strip()
    if new_age:
        try:
            age_val = int(new_age)
            if 10 <= age_val <= 80:
                found["age"] = str(age_val)
        except ValueError:
            print("  Invalid age. Keeping old value.")
    new_course = input(f"  Course [{found['course']}]: ").strip()
    if new_course:
        found["course"] = new_course
    new_marks = input(f"  Marks [{found['marks']}]: ").strip()
    if new_marks:
        try:
            marks_val = float(new_marks)
            if 0 <= marks_val <= 100:
                found["marks"] = str(marks_val)
                found["grade"] = calculate_grade(marks_val)
        except ValueError:
            print("  Invalid marks. Keeping old value.")
    save_data(RECORDS_FILE, students)
    print("  Record updated.")
    return students


def delete_record(students):
    """Find a student by ID and delete after confirmation."""
    print("\n--- Delete Student Record ---")
    student_id = input("  Student ID to delete: ").strip().upper()
    found = None
    for s in students:
        if s["student_id"] == student_id:
            found = s
            break
    if not found:
        print(f"  Error: ID '{student_id}' not found.")
        return students
    print(f"  About to delete: {found['name']} ({found['student_id']})")
    confirm = input("  Are you sure? (yes/no): ").strip().lower()
    if confirm == "yes":
        students.remove(found)
        save_data(RECORDS_FILE, students)
        print("  Deleted.")
    else:
        print("  Delete cancelled.")
    return students


def display_menu():
    """Print the main menu options."""
    print("\n" + "=" * 40)
    print("   STUDENT MANAGEMENT PORTAL")
    print("=" * 40)
    print("  1. View All Students")
    print("  2. Add New Student")
    print("  3. Search Student")
    print("  4. Update Student Record")
    print("  5. Delete Student Record")
    print("  6. Generate Report")
    print("  7. Exit")
    print("=" * 40)


print("view_all_records(), update_record(), delete_record(), display_menu() defined.")

view_all_records(), update_record(), delete_record(), display_menu() defined.


---
## Section 8: Demo — Sample Test Runs

The cells below demonstrate all features without using `input()`.  
These produce the same output you would see running `portal.py` interactively.

In [8]:
# Demo 1: Load data from file
students = load_data(RECORDS_FILE)
print(f"Loaded {len(students)} students from {RECORDS_FILE}")

Loaded 8 students from data/students.csv


In [9]:
# Demo 2: View all records
view_all_records(students)


--- All Student Records ---
  Total: 8 student(s)

  ID       Name                 Age   Course               Marks   Grade
  --------------------------------------------------------------------
  S001     Ali Hassan           20    Computer Science     85      A
  S002     Sara Khan            22    Business Admin       72      B
  S003     Usman Raza           21    Electrical Eng       60      C
  S004     Fatima Noor          19    Computer Science     91      A
  S005     Bilal Ahmed          23    Mathematics          55      D
  S008     maham                20    computer             39.0    F
  S009     batool               28    coding               29.0    F
  S010     Hina Baig            20    Software Eng         77.0    B


In [10]:
# Demo 3: Add a new student (direct call, no input())
new_student = {
    "student_id": "S010",
    "name":       "Hina Baig",
    "age":        "20",
    "course":     "Software Eng",
    "marks":      "77.0",
    "grade":      calculate_grade(77.0)
}
exists = any(s["student_id"] == "S010" for s in students)
if not exists:
    students.append(new_student)
    save_data(RECORDS_FILE, students)
    print(f"Added: {new_student['name']} | Course: {new_student['course']} | Grade: {new_student['grade']}")
else:
    print("S010 already exists.")
view_all_records(students)

S010 already exists.

--- All Student Records ---
  Total: 8 student(s)

  ID       Name                 Age   Course               Marks   Grade
  --------------------------------------------------------------------
  S001     Ali Hassan           20    Computer Science     85      A
  S002     Sara Khan            22    Business Admin       72      B
  S003     Usman Raza           21    Electrical Eng       60      C
  S004     Fatima Noor          19    Computer Science     91      A
  S005     Bilal Ahmed          23    Mathematics          55      D
  S008     maham                20    computer             39.0    F
  S009     batool               28    coding               29.0    F
  S010     Hina Baig            20    Software Eng         77.0    B


In [11]:
# Demo 4: Error handling — invalid input where a number is expected
bad_input = "ninety"
try:
    float(bad_input)
except ValueError:
    print(f"ValueError caught: '{bad_input}' is not a number. Record not saved.")

ValueError caught: 'ninety' is not a number. Record not saved.


In [12]:
# Demo 5: Search by name (partial match)
search_key = "ali"
results = [s for s in students if search_key in s["name"].lower()]
print(f"Searching for '{search_key}' — found {len(results)} result(s):")
if results:
    print_table(results)

Searching for 'ali' — found 1 result(s):
  ID       Name                 Age   Course               Marks   Grade
  --------------------------------------------------------------------
  S001     Ali Hassan           20    Computer Science     85      A


In [13]:
# Demo 6: Missing file — FileNotFoundError handled
missing = load_data("data/does_not_exist.csv")
print(f"Records from missing file: {len(missing)} (handled gracefully, no crash)")

# load_data() auto-creates an empty file when the CSV is missing.
# We delete it here so the demo does not leave a stray file behind.
if os.path.exists("data/does_not_exist.csv"):
    os.remove("data/does_not_exist.csv")
    print("(Cleaned up the auto-created empty file.)")

Note: 'data/does_not_exist.csv' not found. Starting with empty data.
Records from missing file: 0 (handled gracefully, no crash)
(Cleaned up the auto-created empty file.)


In [14]:
# Demo 7: Generate full summary report
generate_report(students)


--- Summary Report ---
      STUDENT MANAGEMENT PORTAL REPORT
  Total Students  : 8
  Average Marks   : 63.50
  Highest Marks   : 91.00
  Lowest Marks    : 29.00

  Grade Distribution:
    Grade A: 2 student(s)
    Grade B: 2 student(s)
    Grade C: 1 student(s)
    Grade D: 1 student(s)
    Grade F: 2 student(s)

  Students Per Course:
    Computer Science: 2 student(s)
    Business Admin: 1 student(s)
    Electrical Eng: 1 student(s)
    Mathematics: 1 student(s)
    computer: 1 student(s)
    coding: 1 student(s)
    Software Eng: 1 student(s)

  Report saved to 'reports/summary_report.txt'.
  Report generation complete.


---
## Demo 8: Grade boundary test run

`calculate_grade()` is a pure function, so it can be tested directly.
The boundaries are A ≥ 80, B ≥ 65, C ≥ 50, D ≥ 40, F < 40 — the test below
checks every boundary value plus one value on each side of it.

In [15]:
# Demo 8: Boundary test cases for calculate_grade()
test_cases = [95, 80, 79, 65, 64, 50, 49, 40, 39, 0]
expected    = ["A", "A", "B", "B", "C", "C", "D", "D", "F", "F"]

print(f"{'Marks':>6} | Expected | Actual | Result")
print("-" * 40)
passed = 0
for marks, exp in zip(test_cases, expected):
    actual = calculate_grade(marks)
    ok = actual == exp
    passed += ok
    print(f"{marks:>6} | {exp:^8} | {actual:^6} | {'PASS' if ok else 'FAIL'}")
print("-" * 40)
print(f"{passed}/{len(test_cases)} boundary tests passed.")

 Marks | Expected | Actual | Result
----------------------------------------
    95 |    A     |   A    | PASS
    80 |    A     |   A    | PASS
    79 |    B     |   B    | PASS
    65 |    B     |   B    | PASS
    64 |    C     |   C    | PASS
    50 |    C     |   C    | PASS
    49 |    D     |   D    | PASS
    40 |    D     |   D    | PASS
    39 |    F     |   F    | PASS
     0 |    F     |   F    | PASS
----------------------------------------
10/10 boundary tests passed.


---
## Section 9: main() — The Entry Point

When you run `python portal.py`, Python calls `main()` automatically.

It does three things in order:
1. Calls `user_login()` — exits if login fails
2. Calls `load_data()` — loads students from CSV
3. Runs a `while True` loop — shows menu, calls the right function, loops until user picks Exit

In [16]:
def main():
    """Entry point — login, load data, then run menu loop."""
    print("\n" + "=" * 40)
    print("   STUDENT MANAGEMENT PORTAL")
    print("=" * 40)

    username, role = user_login()
    if not username:
        print("\n  Exiting portal.")
        return

    students = load_data(RECORDS_FILE)

    while True:
        display_menu()
        choice = input("  Enter choice (1-7): ").strip()

        if choice == "1":
            view_all_records(students)
        elif choice == "2":
            students = add_record(students)
        elif choice == "3":
            search_record(students)
        elif choice == "4":
            students = update_record(students)
        elif choice == "5":
            students = delete_record(students)
        elif choice == "6":
            generate_report(students)
        elif choice == "7":
            print("\n  Goodbye!")
            break
        else:
            print("  Invalid choice. Enter a number from 1 to 7.")


# Note: in this notebook we do not call main() directly
# because it uses input(). Run portal.py in a terminal instead.
print("main() defined. Run portal.py in terminal for the full interactive version.")

main() defined. Run portal.py in terminal for the full interactive version.


---
## Section 10: Test Run Summary

| # | Demo | Feature exercised | Result |
|---|------|-------------------|--------|
| 1 | Demo 1 | `load_data()` — reading the CSV | 7 records loaded |
| 2 | Demo 2 | `view_all_records()` — table output | All records printed |
| 3 | Demo 3 | Add record + `save_data()` + duplicate guard | S010 added (or skipped if already present) |
| 4 | Demo 4 | `ValueError` handling on bad marks input | Error caught, record not saved |
| 5 | Demo 5 | Name search (partial, case-insensitive) | Matching records found |
| 6 | Demo 6 | `FileNotFoundError` handling | Missing file handled, no crash |
| 7 | Demo 7 | `generate_report()` + `finally` block | Stats computed, report saved |
| 8 | Demo 8 | `calculate_grade()` boundary tests | 10/10 expected to pass |

**Reproducing these outputs:** use *Kernel → Restart & Run All* — the notebook
runs top-to-bottom with no errors and regenerates every output cell. The
interactive functions (`user_login`, `add_record`, `update_record`,
`delete_record`, `main`) are left un-executed here because they require
`input()`; run `python portal.py` in a terminal for the fully interactive
version, or `python app.py` for the web version.